## Structured Output

By default a model returns **free text**. That is hard to use in code, because you would have to parse it yourself. **Structured output** makes the model return data that matches a **schema** you define, such as fields, types and allowed values. You get a Python object back instead of a string.

```python
# Free text: hard to use in code
"The movie Inception was directed by Christopher Nolan in 2010."

# Structured: easy to use in code
Movie(title="Inception", director="Christopher Nolan", year=2010)
```

**Use structured output when:**

- You extract data from text (names, dates, amounts, entities).
- You classify input into fixed labels.
- The output feeds another system (a database, an API, another tool).
- You need to validate what the model returned, which matters for guardrails.

---

## Pydantic

**Pydantic** is a Python library that defines data shapes as classes and **validates** data against them. It is the most common way to describe a schema in LangChain.

```python
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """Details about a movie."""
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The name of the director")
    year: int = Field(description="The release year")
```

### What Pydantic gives you

| Feature | What it does |
|---|---|
| **Type checking** | `year: int` rejects data that can't become an int |
| **Coercion** | `"2010"` is converted to `2010` |
| **Validation errors** | A wrong or missing field raises a clear `ValidationError` |
| **Descriptions** | `Field(description=...)` is sent to the model as guidance |
| **JSON schema** | Pydantic converts the class into the JSON schema the model needs |

### Why descriptions matter

The class docstring and each `Field(description=...)` are part of the prompt the model sees. Clear descriptions give better results.

### Common field patterns

```python
from typing import Literal, Optional
from pydantic import BaseModel, Field

class Ticket(BaseModel):
    title: str
    priority: Literal["low", "medium", "high"]        # only these values allowed
    score: int = Field(ge=1, le=5)                     # must be between 1 and 5
    assignee: Optional[str] = None                     # may be missing
    tags: list[str] = Field(default_factory=list)      # a list of strings
```

### Nested models

```python
class Address(BaseModel):
    city: str
    country: str

class Person(BaseModel):
    name: str
    address: Address          # a model inside a model
```

### Custom validation

```python
from pydantic import BaseModel, field_validator

class Review(BaseModel):
    text: str
    rating: int

    @field_validator("rating")
    @classmethod
    def check_rating(cls, v):
        if not 1 <= v <= 5:
            raise ValueError("rating must be between 1 and 5")
        return v
```

### Useful methods

```python
movie = Movie(title="Inception", director="Christopher Nolan", year=2010)

movie.model_dump()          # -> dict
movie.model_dump_json()     # -> JSON string
Movie.model_validate_json('{"title": "Inception", "director": "Nolan", "year": 2010}')
```

---

## Structured output in LangChain

There are two places to use it: on an **agent** and directly on a **model**.

### 1. With an agent: `response_format`

Pass your schema to `create_agent`. The result is in `result["structured_response"]`.

```python
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[],
    response_format=Movie,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about the movie Inception"}]}
)

result["structured_response"]
# Movie(title='Inception', director='Christopher Nolan', year=2010)
```

The agent runs its normal loop (model calls and tool calls). Then it produces the final answer in your schema.

### 2. With a model: `with_structured_output`

Use this when you call a model directly, without an agent.

```python
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.3-70b-versatile")
structured_model = model.with_structured_output(Movie)

movie = structured_model.invoke("Tell me about the movie Inception")
print(movie.title, movie.year)     # a real Movie object, not a string
```

### Which one to use

| Need | Use |
|---|---|
| Agent with tools, final answer in a fixed shape | `create_agent(..., response_format=Schema)` |
| One model call, one structured result | `model.with_structured_output(Schema)` |

---

## Strategies

LangChain has two ways to get structured output from a model.

| Strategy | How it works | Notes |
|---|---|---|
| **`ProviderStrategy`** | Uses the provider's native structured output feature | Most reliable, but only for providers that support it |
| **`ToolStrategy`** | Turns the schema into a **tool** the model calls | Works with any model that supports tool calling |

If you pass a schema directly, LangChain picks the best strategy for the model. You can also choose one yourself:

```python
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy

agent = create_agent(model="openai:gpt-5", tools=[], response_format=ToolStrategy(Movie))
agent = create_agent(model="openai:gpt-5", tools=[], response_format=ProviderStrategy(Movie))
```

### Error handling with `ToolStrategy`

If the model returns data that fails validation, `ToolStrategy` sends the error back to the model so it can try again.

```python
agent = create_agent(
    model="openai:gpt-5",
    tools=[],
    response_format=ToolStrategy(Ticket, handle_errors=True),
)
```

---

## Other schema types

Pydantic is not the only option.

| Schema type | Returns | Validates? |
|---|---|---|
| **Pydantic `BaseModel`** | Pydantic object | Yes (types, constraints, custom rules) |
| **`dataclass`** | dataclass object | Basic |
| **`TypedDict`** | `dict` | No (hints only) |
| **JSON Schema** (`dict`) | `dict` | No |

```python
from typing_extensions import TypedDict

class Movie(TypedDict):
    title: str
    director: str
    year: int
```

**Recommendation:** use Pydantic. It checks the model's output, and that is what you want when you build guardrails.

---

## Getting the raw message too

`include_raw=True` returns the parsed object, the raw `AIMessage`, and any parsing error.

```python
structured_model = model.with_structured_output(Movie, include_raw=True)
out = structured_model.invoke("Tell me about Inception")

out["parsed"]           # Movie object (or None if parsing failed)
out["raw"]              # the original AIMessage (token usage, metadata)
out["parsing_error"]    # the exception, or None
```

---

## Structured output and guardrails

Structured output is a guardrail in itself:

- **Constrains the output:** the model can only answer with the fields and values you allow (`Literal`, `ge`/`le`).
- **Validates it:** bad output fails with a `ValidationError` instead of silently passing through.
- **Blocks free text:** less room for the model to add unwanted content.
- **Feeds later checks:** typed fields are easy to test (`if result.risk == "high": ...`).

```python
class ModerationResult(BaseModel):
    """Safety check on a user message."""
    is_safe: bool = Field(description="True if the message is safe")
    category: Literal["safe", "harassment", "violence", "self_harm", "other"]
    reason: str = Field(description="One sentence explaining the decision")

checker = model.with_structured_output(ModerationResult)
result = checker.invoke("How do I pick a lock?")

if not result.is_safe:
    print("Blocked:", result.category, "-", result.reason)
```

---

## Summary

| Concept | Key point |
|---|---|
| Structured output | The model returns data in your schema, not free text |
| Pydantic | Defines the schema and validates the output |
| `Field(description=...)` | Tells the model what each field means |
| `response_format=` | Structured output on an agent (`result["structured_response"]`) |
| `with_structured_output()` | Structured output on a model |
| `ProviderStrategy` / `ToolStrategy` | Native provider support vs. tool-calling fallback |
| Best practice | Use Pydantic, write clear descriptions, keep schemas small |


In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [4]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Tell me a joke")
response


AIMessage(content='Why did the scarecrow win an award?\n\nBecause he was outstanding in his field! 🌾', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 16, 'total_tokens': 38, 'completion_time': 0.04239373, 'completion_tokens_details': None, 'prompt_time': 0.000833179, 'prompt_tokens_details': None, 'queue_time': 0.047653658, 'total_time': 0.043226909}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0c418-0bba-7401-9438-ac5be89bab5f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 22, 'total_tokens': 38})

In [6]:
## Import Pydantic

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="the director of the movie")
    rating:float=Field(description="The movies rating out of 10")



In [8]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x10dfc1810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10dfc2210>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'descript

In [10]:
model.invoke("Provide detailes about movie inception")

AIMessage(content='**Inception** is a 2010 science fiction action film written and directed by **Christopher Nolan**. It is widely regarded as one of the most complex, visually stunning, and intellectually stimulating films of the 21st century. The movie stars **Leonardo DiCaprio**, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Cillian Murphy, and Marion Cotillard.\n\n### 🎬 Basic Information\n- **Title**: Inception\n- **Director/Writer**: Christopher Nolan\n- **Release Date**: July 16, 2010\n- **Runtime**: 148 minutes\n- **Genre**: Action, Science Fiction, Thriller\n- **Music**: Hans Zimmer (feature prominently in the Oscar-nominated soundtrack)\n- **Studio**: Warner Bros. Pictures\n\n### 📖 Plot Summary\n\n#### The Premise\nThe film explores the concept of **inception**: the process of planting an idea in a person’s subconscious so that they believe it is their own. This is achieved through a technology that allows people to share dreams, where time moves slower in deeper 

In [ ]:
model_with_structure.invoke("Provide detailes about movie inception")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [13]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5")

In [14]:
## Import Pydantic

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="the director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide detailes about movie inception")
response


{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 866, 'prompt_tokens': 113, 'total_tokens': 979, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQYIURh0hK8xfyizpCGAfU3Y2Rrty', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c423-bf4b-7871-9e1b-1969c6d2421b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_tokens': 866, 'tot

### Nested structure

A **nested structure** is a Pydantic model used as the type of a field in another model. Use it when the data has a natural hierarchy, such as a person with an address or an order with items.

```python
from pydantic import BaseModel, Field

class Address(BaseModel):
    city: str = Field(description="City name")
    country: str = Field(description="Country name")

class Person(BaseModel):
    name: str = Field(description="Full name")
    age: int = Field(description="Age in years")
    address: Address = Field(description="Where the person lives")
```

```python
structured_model = model.with_structured_output(Person)
person = structured_model.invoke("Ravi is 28 and lives in Chennai, India.")

person.name             # 'Ravi'
person.address.city     # 'Chennai'
```

### Lists of nested models

Use `list[...]` when a field holds many items of the same shape.

```python
class Item(BaseModel):
    product: str
    quantity: int

class Order(BaseModel):
    customer: str
    items: list[Item]       # many Item objects
```

```python
order = structured_model.invoke("Asha ordered 2 pens and 3 notebooks.")

order.items[0].product   # 'pens'
order.items[1].quantity  # 3
```

### Convert to a dict or JSON

Nested models convert all the way down.

```python
order.model_dump()
# {'customer': 'Asha', 'items': [{'product': 'pens', 'quantity': 2}, {'product': 'notebooks', 'quantity': 3}]}
```

**Tips:**

- Add a `Field(description=...)` to every model and field, including the inner ones.
- Keep nesting shallow. Deep schemas make the model's output less reliable.
- Define the inner model **before** the outer one that uses it.


In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    geres:list[str]
    budget:float | None = Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide detailes about movie inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Tom Berenger', role='Peter Browning'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Pete Postlethwaite', role='Maurice Fischer'), Actor(name='Lukas Haas', role='Nash'), Actor(name='Talulah Riley', role='Blonde')], geres=['Science fiction', 'Action', 'Thriller', 'Heist'], budget=160.0)

## TypedDict Structured Output

A **`TypedDict`** describes the shape of a dictionary: its keys and the type of each value. As a structured output schema, it makes the model return a plain **`dict`** with those keys. The type hints are only a description of the shape. **Python does not check them at runtime**, so nothing validates the model's output and a wrong type or value passes through silently. If you need validation, use Pydantic.

### Basic example

```python
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    """Details about a movie."""
    title: Annotated[str, ..., "Movie title"]
    year: Annotated[int, ..., "Release year"]
    rating: Annotated[float, ..., "Rating out of 10"]

structured_model = model.with_structured_output(Movie)
movie = structured_model.invoke("Tell me about the movie Inception")

movie
# {'title': 'Inception', 'year': 2010, 'rating': 8.8}

movie["title"]        # access with keys, not movie.title
```

**Annotated pattern:** `Annotated[type, default, "description"]`

| Part | Meaning |
|---|---|
| `type` | The value type (`str`, `int`, `float`, `list[str]`, ...) |
| `...` | Field is **required** (use `None` for an optional field) |
| `"description"` | Sent to the model as guidance for that field |

The class docstring and the descriptions are part of the prompt the model sees, so write them clearly.

### Optional, Literal and list fields

```python
from typing import Literal, Optional

class Ticket(TypedDict):
    title: Annotated[str, ..., "Short summary of the issue"]
    priority: Annotated[Literal["low", "medium", "high"], ..., "How urgent the issue is"]
    assignee: Annotated[Optional[str], None, "Person assigned, if any"]   # optional
    tags: Annotated[list[str], ..., "Labels for the ticket"]
```

### Nested TypedDict

Define the inner type **first**.

```python
class Item(TypedDict):
    product: Annotated[str, ..., "Product name"]
    quantity: Annotated[int, ..., "Number of units"]

class Order(TypedDict):
    customer: Annotated[str, ..., "Customer name"]
    items: Annotated[list[Item], ..., "Items in the order"]

order = model.with_structured_output(Order).invoke("Asha ordered 2 pens and 3 notebooks.")

order["items"][0]["product"]     # 'pens'
```

### Use it with an agent

```python
from langchain.agents import create_agent

agent = create_agent(model="openai:gpt-5", tools=[], response_format=Movie)
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me about Inception"}]})

result["structured_response"]    # a dict
```

### No runtime validation

`TypedDict` hints are **not enforced when the code runs**. A wrong type passes through without an error.

```python
class Movie(TypedDict):
    title: str
    year: int

bad: Movie = {"title": "Inception", "year": "not a number"}   # no error
print(bad["year"])                                            # 'not a number'
```

- Type checkers (Pylance, mypy) flag this in your editor. Python itself does not.
- There is no coercion (`"2010"` stays a string), no constraints (`ge`, `le`) and no custom validators.
- If the model returns a wrong type, you must check it yourself, or use Pydantic instead.

### When to use TypedDict

| Use it when | Avoid it when |
|---|---|
| You want a plain dict for JSON, an API or a DataFrame | You must trust the output (guardrails) |
| The schema is simple | You need constraints or custom rules |
| You don't want a Pydantic dependency | You want typed objects with dot access |

### Summary

| Point | TypedDict |
|---|---|
| Returns | `dict` |
| Access | `result["key"]` |
| Validation | None at runtime |
| Description syntax | `Annotated[type, default, "text"]` |
| Required / optional | `...` / `None` |
| Nesting | Yes, inner type defined first |


In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie released"]
    direcotor: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [20]:
model_with_typed_dict=model.with_structured_output(MovieDict)
response=model_with_typed_dict.invoke("Please provide the details of the movie intersteller")
response

{'title': 'Interstellar',
 'year': 2014,
 'direcotor': 'Christopher Nolan',
 'rating': 8.6}

In [21]:


class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    geres:list[str]
    budget:float | None = Field(None, description="Budget in millions USD")

model_with_typed_dict=model.with_structured_output(MovieDetails)

response = model_with_typed_dict.invoke("Provide detailes about movie inception")
response


{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Marion Cotillard', 'role': 'Mal Cobb'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Michael Caine', 'role': 'Miles'},
  {'name': 'Dileep Rao', 'role': 'Yusuf'},
  {'name': 'Tom Berenger', 'role': 'Peter Browning'}],
 'geres': ['Science Fiction', 'Action', 'Thriller', 'Heist'],
 'budget': 160000000}

In [22]:
model_with_typed_dict.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [23]:
model.profile

{'name': 'GPT-5',
 'release_date': '2025-08-07',
 'last_updated': '2025-08-07',
 'open_weights': False,
 'max_input_tokens': 272000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high']}

## Dataclass Structured Output

A **`dataclass`** is a standard Python class for holding data. Python generates `__init__`, `__repr__` and `__eq__` for you. As a structured output schema, it makes LangChain return an **instance of your dataclass**, so you access fields with dot notation (`movie.title`).

**Validation and restrictions:** a dataclass does **not** validate or restrict anything by itself. Its type hints are ignored at runtime, so `Movie("A", 15, "horror")` is accepted without an error. The checking happens when it is used as a **LangChain schema** (for example `create_agent(response_format=Movie)`). LangChain parses the model's output with a Pydantic `TypeAdapter`, which enforces the schema:

| Rule | Example | If the model breaks it |
|---|---|---|
| Type check and conversion | `year: int` | `"2010"` becomes `2010`, `"abc"` raises an error |
| Required field | `title: str` (no default) | Missing field raises an error |
| Allowed values | `Literal["drama", "comedy"]` | `"horror"` raises an error |
| Range limit | `Annotated[float, Field(ge=0, le=10)]` | `15` raises an error |

So the restrictions live in `Literal` and `Field(...)`, and LangChain enforces them on the model's output only.

### Basic example

```python
from dataclasses import dataclass
from typing import Annotated
from pydantic import Field

@dataclass
class Movie:
    """Details about a movie."""
    title: Annotated[str, Field(description="Movie title")]
    year: Annotated[int, Field(description="Release year")]
    rating: Annotated[float, Field(description="Rating out of 10", ge=0, le=10)]
```

Use it with an agent:

```python
from langchain.agents import create_agent

agent = create_agent(model="openai:gpt-5", tools=[], response_format=Movie)
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me about Inception"}]})

result["structured_response"]
# Movie(title='Inception', year=2010, rating=8.8)

result["structured_response"].title     # dot access, like Pydantic
```

### Adding descriptions

| Where | How |
|---|---|
| Whole schema | The class **docstring** |
| One field | `Annotated[type, Field(description="...")]` (the `Field` comes from Pydantic) |

The docstring and descriptions are sent to the model as guidance. Without them, the model only sees field names and types.

### Optional, Literal, list and default fields

Fields **without a default are required**. Fields **with a default are optional**.

```python
from dataclasses import dataclass, field
from typing import Literal, Optional

@dataclass
class Ticket:
    """A support ticket."""
    title: str
    priority: Literal["low", "medium", "high"]     # only these values allowed
    assignee: Optional[str] = None                 # optional
    tags: list[str] = field(default_factory=list)  # optional list (use field(), not = [])
```

**Rule:** required fields must come **before** fields with defaults, or Python raises an error.

### Nested dataclasses

Define the inner one **first**.

```python
@dataclass
class Item:
    product: str
    quantity: int

@dataclass
class Order:
    """An order."""
    customer: str
    items: list[Item]

order = agent.invoke(...)["structured_response"]
order.items[0].product          # 'pens'
```

### Is it validated?

It depends **where** the dataclass is used.

| Situation | Validated? |
|---|---|
| LangChain agent (`response_format=`) | **Yes.** LangChain parses the model's output with a Pydantic `TypeAdapter`, so it checks types, required fields, `Literal` values and `Field` limits, and converts `"2"` to `2` |
| You create it yourself: `Movie("Inception", "abc")` | **No.** A plain dataclass never checks types at runtime |

```python
Movie(title="Inception", year="abc", rating=15)   # no error, wrong type and out-of-range value accepted

from pydantic import TypeAdapter
TypeAdapter(Movie).validate_python({"title": "x", "year": "abc", "rating": 15})
# ValidationError: year - Input should be a valid integer
#                  rating - Input should be less than or equal to 10
```

So the validation comes from **LangChain and Pydantic**, not from `@dataclass` itself.

### Convert to a dict

```python
from dataclasses import asdict

asdict(order)
# {'customer': 'Asha', 'items': [{'product': 'pens', 'quantity': 2}, ...]}
```

### Comparison

| | `TypedDict` | `dataclass` | Pydantic `BaseModel` |
|---|---|---|---|
| Returns | `dict` | Object | Object |
| Access | `x["key"]` | `x.key` | `x.key` |
| Validation when you build it yourself | No | No | **Yes** |
| Validation of LangChain agent output | No | Yes (via `TypeAdapter`) | Yes |
| Description syntax | `Annotated[t, ..., "text"]` | `Annotated[t, Field(description="text")]` | `Field(description="text")` |
| Convert to dict | already a dict | `asdict(x)` | `x.model_dump()` |

**Use a dataclass when:**

- You want an object with dot access but don't need Pydantic as your main tool.
- The class is also used in normal app code (as a plain data holder).

**Use Pydantic when:**

- You need validation everywhere, not only for LangChain output.
- You want validators, `model_dump()` and JSON helpers.


In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str = Field(description="The name of the perosn")
    email:str = Field(description="The email address of the perosn")
    phone:str = Field(description="The phone number of the perosn")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Auto-select providerStrategy
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract info from : john Doe, john@example.com, (555)123-4567"}]
})

result
# print(result["structured_response"])


{'messages': [HumanMessage(content='Extract info from : john Doe, john@example.com, (555)123-4567', additional_kwargs={}, response_metadata={}, id='14468bf5-ff42-41ee-ade1-87cd0e60a3a6'),
  AIMessage(content='{"name":"john Doe","email":"john@example.com","phone":"(555)123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 291, 'prompt_tokens': 208, 'total_tokens': 499, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQZ1sJX1TLUBwkP4xjlV3Nk5ko84c', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c44e-b74b-7d91-8762-7c16f5b500de-0',

In [32]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name:str = Field(description="The name of the perosn")
    email:str = Field(description="The email address of the perosn")
    phone:str = Field(description="The phone number of the perosn")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Auto-select providerStrategy
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract info from : john Doe, john@example.com, (555)123-4567"}]
})

result["structured_response"]
# print(result["structured_response"])


{'name': 'john Doe', 'email': 'john@example.com', 'phone': '(555)123-4567'}

In [33]:
## Data class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name:str
    emai:str
    phone:str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract info from : john Doe, john@example.com, (555)123-4567"}]
})

result["structured_response"]

ContactInfo(name='john Doe', emai='john@example.com', phone='(555)123-4567')